# Chatbot de PLN para campañas y cierre de ventas — Fhalconfood Globasupplies Cia Ltda
### Asistente de chat que atiende consultas de clientes HORECA, responde de forma automática y deriva a un asesor humano cuando el cliente quiere comprar

**Proyecto de Titulación — Maestría en Inteligencia Artificial Aplicada (UDLA) · AI Marketing Hub**
**Integrantes:** Peter Cañizares, Henry Mantilla
**Caso de uso:** Fhalconfood Globsupplies Cia Ltda

## 2. Diseño de la solución técnica

**Arquitectura funcional (diagrama de flujo):**

```
Campaña (WhatsApp / redes)  →  Cliente HORECA escribe al chatbot
        ↓
Preprocesamiento del mensaje
        ↓
PLN: Clasificación de intención (BETO afinado) + NER + Búsqueda semántica (embeddings)
        ↓
Consulta al catálogo de productos (validado, no generado por el modelo)
        ↓
Respuesta automática  ──►  consultas informativas (producto, precio, stock, entrega y pago)
        │
        └──►  Intención de compra / pide un asesor / el bot no entiende
                      ↓
              Registro del lead + derivación a ASESOR HUMANO (cierre de la venta)
```

**Técnicas de PLN utilizadas:**

| Técnica | Uso en este prototipo |
|---|---|
| Clasificación de texto (fine-tuning) | BETO afinado sobre un dataset propio de intenciones comerciales del cliente HORECA |
| Reconocimiento de Entidades Nombradas (NER) | Extraer el nombre de la persona y del negocio (ORG) y detectar códigos de producto mediante regex |
| Embeddings / búsqueda semántica (SBERT) | Encontrar el producto que mejor coincide con una descripción libre (ej. "harina para panadería") |
| Generación controlada de respuestas | Plantillas que combinan datos reales del catálogo, evitando que el modelo "invente" precios o stock |
| Derivación a humano (handoff) | Cuando hay intención de compra, el cliente pide un asesor o el bot falla dos veces seguidas |

**Herramientas, librerías y frameworks:** Hugging Face Transformers (`AutoModelForSequenceClassification`, `Trainer`, `pipeline`), Sentence-Transformers (embeddings), Gradio (interfaz conversacional), scikit-learn y seaborn (evaluación y visualización).

**Infraestructura de ejecución:** Google Colab, usando un acelarador GPU para poder obetener mejores resultados.

**Nota sobre los datos.** El catálogo, los precios y las condiciones comerciales de este notebook son una base que nos inventamos. Antes de usarlo con clientes reales deben reemplazarse por los datos oficiales de Fhalconfood.

## 3. Implementación del prototipo

### 3.1 Instalación de dependencias

In [ ]:
!pip install -q transformers[torch] sentence-transformers gradio scikit-learn matplotlib seaborn accelerate

In [ ]:
# Librerías utilizadas en todo el notebook
import re
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

from typing import Dict, Any
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, pipeline, set_seed
)
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Librerías cargadas. Device:", DEVICE)

### 3.2 Carga de modelos base

- **NER:** `mrm8488/bert-spanish-cased-finetuned-ner` — detecta personas, organizaciones (el negocio del cliente), lugares y fechas mencionadas en el mensaje.
- **Embeddings multilingües:** `paraphrase-multilingual-MiniLM-L12-v2`permite comparar semánticamente la descripción libre del cliente contra las descripciones de los productos del catálogo.

El clasificador de intención se afina (fine-tuning) con **BETO** (`dccuchile/bert-base-spanish-wwm-uncased`) sobre un dataset propio.

In [ ]:
ner = pipeline(
    "token-classification",
    model="mrm8488/bert-spanish-cased-finetuned-ner",
    aggregation_strategy="simple",
)

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Modelos base cargados correctamente.")

### 3.3 Fine-tuning de BETO para clasificación de intención

 Al no contar todavía con conversaciones reales de Fhalconfood, se construyó un dataset hipotetico de mensajes típicos de WhatsApp de clientes HORECA. Las intenciones incluyen dos que activan la derivación a un asesor humano (`intencion_compra` y `hablar_con_asesor`) y una clase `otro` con mensajes fuera de dominio.

In [ ]:
datos_intenciones = {
    "saludo": [
        "hola",
        "buenas",
        "buenos días",
        "buenas tardes",
        "buenas noches",
        "hola, cómo están",
        "hey",
        "qué tal",
        "hola buen día",
        "saludos",
        "hola, quisiera preguntar algo",
        "buenas, disculpen la hora",
        "hola, buenas",
        "hola! vi su publicidad",
        "muy buenos días",
        "hola, una consulta",
        "hola equipo",
        "buen día",
        "hola, tienen un momento?",
        "hola, los vi en Instagram"
    ],
    "despedida": [
        "gracias",
        "muchas gracias",
        "gracias por la info",
        "listo, gracias",
        "hasta luego",
        "nos vemos",
        "chao",
        "adiós",
        "bye",
        "eso era todo, gracias",
        "perfecto, gracias por todo",
        "gracias, que tengan buen día",
        "ok gracias, hablamos luego",
        "gracias, saludos",
        "listo eso es todo",
        "vale, gracias por la ayuda",
        "de acuerdo, gracias",
        "gracias, quedo atento",
        "eso sería todo por ahora",
        "gracias, buen día"
    ],
    "consultar_producto": [
        "qué productos manejan",
        "tienen harina para panadería",
        "qué quesos tienen",
        "me pueden enviar el catálogo",
        "qué marcas de aceite venden",
        "tienen salsas para restaurante",
        "qué presentaciones tiene el queso mozzarella",
        "de cuántos kilos viene el saco de harina",
        "qué incluye la promoción de la campaña",
        "tienen productos congelados",
        "qué tipos de papa prefrita venden",
        "cuáles son los productos de la campaña",
        "qué más tienen para panaderías",
        "tienen insumos para catering",
        "el PRD-101 qué presentación tiene",
        "qué productos recomiendan para un restaurante",
        "manejan aceite para freidoras",
        "tienen lácteos para hoteles",
        "qué viene en la caja",
        "cuéntame de sus productos"
    ],
    "consultar_precio": [
        "cuánto cuesta el PRD-101",
        "cuál es el precio de la harina",
        "cuánto vale el saco",
        "cuánto cuesta el queso mozzarella",
        "qué precio tiene el aceite",
        "cuánto cobran por el PRD-102",
        "el aceite en cuánto está",
        "cuánto sale la caja",
        "cuál es el valor por kilo",
        "precio de la papa porfa",
        "me pueden pasar la lista de precios",
        "cuánto cuesta cada unidad",
        "qué valor tiene el galón",
        "el PRD-104 en cuánto está",
        "cuánto piden por la caneca",
        "necesito saber el precio",
        "me dicen el costo por saco",
        "cuál sería el precio por mayor",
        "cuánto cuesta en la promoción",
        "qué precio manejan para restaurantes"
    ],
    "consultar_stock": [
        "hay stock del PRD-101",
        "tienen disponible la harina",
        "sigue disponible el queso",
        "ya se agotó",
        "tienen inventario de aceite",
        "está disponible la papa prefrita",
        "hay existencias",
        "cuándo llega más producto",
        "tienen en bodega el PRD-103",
        "aún queda de la promoción",
        "les queda stock",
        "hay disponibilidad para esta semana",
        "se acabó el aceite",
        "tienen para entrega inmediata",
        "el PRD-104 está disponible",
        "todavía tienen la salsa",
        "cuántas unidades tienen",
        "hay producto disponible hoy",
        "tienen suficiente para un pedido grande",
        "reponen pronto el queso"
    ],
    "consultar_condiciones": [
        "cuál es el pedido mínimo",
        "hacen entregas a domicilio",
        "cuánto demora la entrega",
        "entregan en el norte de Quito",
        "qué formas de pago aceptan",
        "dan crédito a restaurantes",
        "aceptan transferencia",
        "cobran el envío",
        "en cuántos días llega el pedido",
        "entregan fines de semana",
        "hay descuento por volumen",
        "facturan con RUC",
        "cuál es el horario de entrega",
        "tienen crédito a 30 días",
        "cuánto es el mínimo para el envío gratis",
        "pagan contra entrega",
        "hasta qué hora reciben pedidos",
        "entregan a hoteles fuera de Quito",
        "qué pasa si el producto llega dañado",
        "aceptan tarjeta de crédito"
    ],
    "intencion_compra": [
        "quiero hacer un pedido",
        "me interesa comprar",
        "necesito 10 sacos de harina",
        "quiero comprar el PRD-101",
        "hagamos el pedido",
        "quiero ordenar para mi restaurante",
        "me gustaría realizar una compra",
        "necesito abastecer mi panadería",
        "quiero 5 cajas de papa prefrita",
        "quisiera hacer una orden para esta semana",
        "vamos a comprar, cómo procedo",
        "lo quiero, cómo hago el pedido",
        "necesito que me envíen 20 kilos de queso",
        "quiero cerrar el pedido",
        "me llevo el PRD-102",
        "sí, quiero comprar la promoción",
        "hagan el pedido de la harina por favor",
        "necesito pedir para el fin de semana",
        "quiero empezar a comprarles",
        "anótame un pedido de aceite"
    ],
    "hablar_con_asesor": [
        "quiero hablar con un asesor",
        "me pueden comunicar con una persona",
        "necesito hablar con un vendedor",
        "pásame con un humano",
        "quiero que me llame un ejecutivo",
        "comuníquenme con alguien del equipo comercial",
        "prefiero hablar con una persona",
        "hay algún asesor disponible",
        "quiero negociar el precio",
        "necesito una cotización personalizada",
        "puede llamarme un representante",
        "quiero hablar con el gerente de ventas",
        "tengo un reclamo de un pedido",
        "necesito atención personalizada",
        "me pueden llamar por teléfono",
        "con quién puedo hablar para un contrato",
        "pásame con ventas",
        "quiero una reunión con un asesor",
        "necesito que alguien me atienda",
        "quiero tratar un tema con una persona real"
    ],
    "otro": [
        "qué clima hace hoy",
        "cuál es la capital de Ecuador",
        "cómo está el tráfico",
        "recomiéndame un restaurante",
        "qué hora es",
        "cuéntame un chiste",
        "cómo te llamas",
        "eres un robot",
        "qué opinas del fútbol",
        "cuál es tu color favorito",
        "puedes ayudarme con matemáticas",
        "qué día es hoy",
        "cómo se hace una torta",
        "cuál es tu comida favorita",
        "hazme un resumen de una noticia"
    ]
}

filas = [(texto, intencion) for intencion, ejemplos in datos_intenciones.items() for texto in ejemplos]
df_intenciones = pd.DataFrame(filas, columns=["texto", "intencion"])
print(f"Total de ejemplos: {len(df_intenciones)}")
print(df_intenciones["intencion"].value_counts())